## **Query ChEMBL Database Webresource Client for Approved Small Molecule Drugs**

An alternative to the postgres database is to directly query ChEMBL using [chembl_webresource_client](https://github.com/chembl/chembl_webresource_client) which is ChEMBL's official python package to interface with ChEMBL.

The purpose of this notebook is to repeat the tasks in the Postgres notebook.

Hint: Install chembl_webresource_client

```python
    pixi add --pypi chembl_webresource_client
```

In [1]:
# Import modules
import chembl_webresource_client
from chembl_webresource_client.new_client import new_client
import pandas as pd

# Expand to see all columns
pd.set_option("display.max_columns", None)

# Print versions
print(f"Pandas Version: {pd.__version__}")
print(f"Chembl Webresource Client Version: {chembl_webresource_client.__version__}")

Pandas Version: 3.0.2
Chembl Webresource Client Version: development


### **Explore Available Tables in ChEMBL**

The tables are now attributes in the new_client object.

In [2]:
dir(new_client)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'activity',
 'activity_supplementary_data_by_activity',
 'assay',
 'assay_class',
 'atc_class',
 'binding_site',
 'biotherapeutic',
 'cell_line',
 'chembl_id_lookup',
 'chembl_release',
 'compound_record',
 'compound_structural_alert',
 'description',
 'document',
 'document_similarity',
 'drug',
 'drug_indication',
 'drug_warning',
 'go_slim',
 'image',
 'mechanism',
 'metabolism',
 'molecule',
 'molecule_form',
 'official',
 'organism',
 'protein_classification',
 'similarity',
 'source',
 'substructure',
 'target',
 'target_component',
 'target_relation',
 

### **Determine Available Columns in the Molecule Object**

All the interesting compound information is part of the molecule attribute.

In [3]:
molecule_client = new_client.molecule
molecule_structures = pd.DataFrame([molecule_client[0]["molecule_structures"]]).columns
molecule_columns = pd.DataFrame([molecule_client[0]]).columns
print(f"{molecule_structures}")
print(f"{molecule_columns}")

Index(['canonical_smiles', 'molfile', 'standard_inchi', 'standard_inchi_key'], dtype='str')
Index(['atc_classifications', 'availability_type', 'biotherapeutic',
       'black_box_warning', 'chemical_probe', 'chirality', 'cross_references',
       'dosed_ingredient', 'first_approval', 'first_in_class', 'helm_notation',
       'inorganic_flag', 'max_phase', 'molecule_chembl_id',
       'molecule_hierarchy', 'molecule_properties', 'molecule_structures',
       'molecule_synonyms', 'molecule_type', 'natural_product', 'oral',
       'orphan', 'parenteral', 'polymer_flag', 'pref_name', 'prodrug',
       'structure_type', 'therapeutic_flag', 'topical', 'usan_stem',
       'usan_stem_definition', 'usan_substem', 'usan_year', 'veterinary',
       'withdrawn_flag'],
      dtype='str')


In [4]:
chembl_columns = [
    "canonical_smiles",
    "chembl_id",
    "pref_name",
    "max_phase",
    "therapeutic_flag",
    "dosed_ingredient",
    "structure_type",
    "molecule_type",
    "first_approval",
    "oral",
    "parenteral",
    "topical",
    "black_box_warning",
    "natural_product",
    "first_in_class",
    "chirality",
    "prodrug",
    "inorganic_flag",
    "usan_year",
    "availability_type",
    "usan_stem",
    "polymer_flag",
    "usan_substem",
    "usan_stem_definition",
    "withdrawn_flag",
    "chemical_probe",
    "orphan",
    "veterinary",
    "atc_classifications",
]
update_columns = {
    "molecule_chembl_id": "chembl_id",
    "molecule_structures.canonical_smiles": "canonical_smiles",
}
molecules_list = molecule_client.filter(max_phase=4)
chembl_df = pd.json_normalize(molecules_list)
chembl_df.rename(columns=update_columns, inplace=True)
chembl_df = chembl_df.loc[:, chembl_columns]
print(chembl_df.shape)
chembl_df.head()

(4005, 29)


,canonical_smiles,chembl_id,pref_name,max_phase,therapeutic_flag,dosed_ingredient,structure_type,molecule_type,first_approval,oral,parenteral,topical,black_box_warning,natural_product,first_in_class,chirality,prodrug,inorganic_flag,usan_year,availability_type,usan_stem,polymer_flag,usan_substem,usan_stem_definition,withdrawn_flag,chemical_probe,orphan,veterinary,atc_classifications
0,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC,CHEMBL2,PRAZOSIN,4.0,True,False,MOL,Small molecule,1976.0,True,False,False,0,1,0,2,0,0,1968.0,1.0,-azosin,0,-azosin,antihypertensives (prazosin type),False,0,0,0,[C02CA01]
1,CN1CCC[C@H]1c1cccnc1,CHEMBL3,NICOTINE,4.0,True,True,MOL,Small molecule,1984.0,True,False,True,0,1,0,1,0,0,1985.0,2.0,NaN,0,NaN,NaN,False,0,0,0,[N07BA01]
2,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23,CHEMBL4,OFLOXACIN,4.0,True,True,MOL,Small molecule,1990.0,True,True,True,1,0,0,0,0,0,1984.0,1.0,-oxacin,0,-oxacin,antibacterials (quinolone derivatives),False,0,0,0,"[J01MA01, S02AA16, S01AE01]"
3,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21,CHEMBL5,NALIDIXIC ACID,4.0,True,True,MOL,Small molecule,1964.0,True,False,False,0,1,0,2,0,0,1962.0,0.0,nal-,0,nal-,narcotic agonists/antagonists (normorphine type),False,0,0,0,[J01MB02]
4,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1,CHEMBL6,INDOMETHACIN,4.0,True,True,MOL,Small molecule,1965.0,True,True,True,1,1,0,2,0,0,1963.0,1.0,NaN,0,NaN,NaN,False,0,0,0,"[C01EB03, M02AA23, M01AB51, S01BC01, M01AB01]"


In [5]:
chembl_df.to_csv("chembl_approved_small_molecule_drugs.csv", index=False)